In [1]:
%load_ext autoreload
%autoreload 2
import os
os.environ["MUJOCO_GL"] = "egl"

In [2]:
import numpy as np

In [19]:
import argparse
import pathlib
import sys

import numpy as np
import torch
#from omegaconf import OmegaConf

import sys

sys.path.append("/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch")
import dreamer as dreamer_main
import models as dreamer_models
import tools as dreamer_tools
import ruamel.yaml as yaml

In [4]:
import pickle

with open(
        "/home/hgf_hmgu/hgf_gib4562/tdmpc2/analysis/outputs_pixel_vs_state/vars_all_pixel_vs_state.pkl",
        "rb") as f:
    vars_all_pixel_vs_state = pickle.load(f)

In [5]:
vars_all_pixel_vs_state.keys()

dict_keys(['z_tdmpc2_state', 'z_tdmpc2_pixel', 'obs', 'position', 'cos(pole_angle)', 'sin(pole_angle)', 'cart_velocity', 'pole_angular_velocity', 'rewards', 'actions'])

In [6]:
vars_all_pixel_vs_state.keys()

position = np.concatenate(vars_all_pixel_vs_state['position'], axis=0)
cos_angle = np.concatenate(vars_all_pixel_vs_state['cos(pole_angle)'], axis=0)
sin_angle = np.concatenate(vars_all_pixel_vs_state['sin(pole_angle)'], axis=0)
cart_velocity = np.concatenate(vars_all_pixel_vs_state['cart_velocity'], axis=0)
pole_velocity = np.concatenate(vars_all_pixel_vs_state['pole_angular_velocity'],
                               axis=0)
action = np.concatenate(vars_all_pixel_vs_state['actions'], axis=0)

In [7]:
obs_batch2 = {}
obs_batch2['position'] = np.concatenate([position, cos_angle, sin_angle],
                                        axis=1)
obs_batch2['velocity'] = np.concatenate([cart_velocity, pole_velocity], axis=1)
obs_batch2['action'] = action

obs_batch2['is_first'] = np.zeros((obs_batch2['position'].shape[0], 1),
                                  dtype=np.float32)
obs_batch2['is_first'][::500] = 1

for k in obs_batch2:
    obs_batch2[k] = np.expand_dims(obs_batch2[k], axis=0)

obs_batch2['position'].shape, obs_batch2['velocity'].shape, obs_batch2[
    'action'].shape


((1, 7500, 3), (1, 7500, 2), (1, 7500, 1))

In [8]:
# Real Dreamer obs_batch from DMC cartpole_swingup (vision + proprio)
import numpy as np
import torch

from envs import dmc
import envs.wrappers as wrappers

T = 100  # rollout length

# Create env (matches Dreamer DMC defaults)
env = dmc.DeepMindControl("cartpole_swingup",
                          action_repeat=2,
                          size=(64, 64),
                          seed=0)
env = wrappers.NormalizeActions(env)

obs_list = []
action_list = []
reward_list = [0.0]
discount_list = [1.0]

obs = env.reset()
obs_list.append(obs)

for _ in range(T - 1):
    action = env.action_space.sample()
    next_obs, reward, done, info = env.step(action)
    action_list.append(action)
    reward_list.append(reward)
    discount_list.append(float(info.get("discount", 1.0)))
    obs_list.append(next_obs)
    if done:
        break

# Build obs_batch with batch dimension
obs_batch = {}
for key in obs_list[0].keys():
    obs_batch[key] = np.stack([o[key] for o in obs_list], axis=0)[None, ...]

# Actions: zero for t=0, then executed actions
action_dim = env.action_space.shape[0]
actions = np.zeros((len(obs_list), action_dim), dtype=np.float32)
if action_list:
    actions[1:1 + len(action_list)] = np.stack(action_list, axis=0)
obs_batch["action"] = actions[None, ...]

# Rewards/discounts aligned with obs_list
#obs_batch["reward"] = np.array(reward_list, dtype=np.float32)[None, ...]
#obs_batch["discount"] = np.array(discount_list, dtype=np.float32)[None, ...]

# Use the same actions downstream for rollouts
#actions = obs_batch["action"]
#actions = torch.tensor(actions, dtype=torch.float32)


libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card3: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card0: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card3: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card2: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card1: Permission denied

libEGL warning: egl: failed to create dri2 screen
libEGL warning: failed to open /dev/dri/card0: Permission denied

/home/hgf_hmgu/hgf_gib4562/miniconda3/envs/dreamerv3/lib/python3.11/site

In [9]:
obs_batch['position'].shape, obs_batch['velocity'].shape, obs_batch[
    'action'].shape


((1, 100, 3), (1, 100, 2), (1, 100, 1))

In [10]:
def _parse_dreamer_config(configs_path, config_names, overrides):
    configs = yaml.safe_load(pathlib.Path(configs_path).read_text())

    def recursive_update(base, update):
        for key, value in update.items():
            if isinstance(value, dict) and key in base:
                recursive_update(base[key], value)
            else:
                base[key] = value

    name_list = ["defaults", *config_names] if config_names else ["defaults"]
    defaults = {}
    for name in name_list:
        recursive_update(defaults, configs[name])

    overrides_map = {}
    for item in overrides:
        if item.startswith("--"):
            item = item[2:]
        if "=" in item:
            key, value = item.split("=", 1)
            if key not in defaults:
                raise KeyError(f"Unknown config key '{key}' in override.")
            cast = dreamer_tools.args_type(defaults[key])
            overrides_map[key] = cast(value)
        else:
            if item not in defaults:
                raise KeyError(f"Unknown config key '{item}' in override.")
            overrides_map[item] = True

    merged = {**defaults, **overrides_map}
    return argparse.Namespace(**merged)

In [11]:
def load_dreamer_world_model(
        configs_path="/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/configs.yaml",
        config_names=["dmc_proprio"],
        overrides=["task=dmc_cartpole_swingup"],
        ckpt_path='/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/logdir/dmc_cartpole_swingup_srun/latest.pt',
        env=None,
        device='cuda:0'):
    cfg = _parse_dreamer_config(
        configs_path,
        config_names,
        overrides,
    )
    cfg.num_actions = 1
    assert env is not None, "You must provide an environment with a valid observation_space."
    wm = dreamer_models.WorldModel(env.observation_space,
                                   None,
                                   step=0,
                                   config=cfg).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    agent_state = ckpt.get("agent_state_dict", ckpt)
    wm_state = {
        k[len("_wm."):]: v
        for k, v in agent_state.items()
        if k.startswith("_wm.")
    }
    wm.load_state_dict(wm_state, strict=False)
    wm.eval()
    return wm, cfg


# Example usage (you must provide 'env'):
wm, cfg = load_dreamer_world_model(env=env)

Encoder CNN shapes: {}
Encoder MLP shapes: {'position': (3,), 'velocity': (2,)}
Decoder CNN shapes: {}
Decoder MLP shapes: {'position': (3,), 'velocity': (2,)}


/home/hgf_hmgu/hgf_gib4562/tdmpc2/dreamerv3-torch/tools.py:747: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/scratch/slurm_tmpdir/job_1604248/ipykernel_18804/1745230561.py:19: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you

Optimizer model_opt has 16428293 variables.


In [ ]:
def get_feats_all(wm, obs_batch):
    data = wm.preprocess(obs_batch)
    embed = wm.encoder(data)
    post, _ = wm.dynamics.observe(embed, data["action"], data["is_first"])
    feats = wm.dynamics.get_feat(post)  # (B, T, feat_dim)

    probs = torch.softmax(post["logit"], dim=-1)  # (B,T,stoch,discrete)
    state_soft = {**post, "stoch": probs}
    feat_soft = wm.dynamics.get_feat(state_soft)
    return feats, feat_soft


#Example usage:
feats_old, feat_soft = get_feats_all(wm, obs_batch2)
feats = feats.squeeze().detach().cpu().numpy()
feat_soft = feat_soft.squeeze().detach().cpu().numpy()
print(feats.shape, feat_soft.shape)

(7500, 1536) (7500, 1536)


In [16]:
vars_all_pixel_vs_state['z_dreamer_onehot'] = feats
vars_all_pixel_vs_state['z_dreamer_onehot_stochastic'] = feats[:, :1024]
vars_all_pixel_vs_state['z_dreamer_onehot_deterministic'] = feats[:, 1024:]
vars_all_pixel_vs_state['z_dreamer_soft'] = feat_soft
vars_all_pixel_vs_state['z_dreamer_soft_stochastic'] = feat_soft[:, :1024]
vars_all_pixel_vs_state['z_dreamer_soft_deterministic'] = feat_soft[:, 1024:]

In [17]:
import pickle

In [18]:
with open(
        "/home/hgf_hmgu/hgf_gib4562/tdmpc2/analysis/outputs_tdmpc2_vs_dreamer/vars_tdmpc2_vs_dreamer.pkl",
        "wb") as f:
    pickle.dump(vars_all_pixel_vs_state, f)